In [32]:
%load_ext autoreload
%autoreload 2

import os,sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import util as yu
from util import *
import util_Nsgm as yu2

yu.setpath('analysis_2ptGEVP')

ens='b'
tfs=[8,10,12,14,16,18,20]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
[c2ptM,tf2c3ptM,c2ptCorrDic_NJN]=yu.load_pkl_reg('data',pathlabel='processData')

In [31]:
# 2pt GEVP

dt=2
t0s=np.arange(0,22-dt)
ts=t0s+dt

t=[yu.GEVP(c,t0s,tList=ts) for c in c2ptM]
evals=np.array([eval for eval,evec in t])
evecs=np.array([evec for eval,evec in t])
evecsInv=np.linalg.inv(evecs)
# print(evals.shape,evecs.shape)

fig,axs=yu.getFigAxs(3,2,sharex='col',sharey='row')
ax=axs[0,0]
n2xmin_plt=[1,1]; n2xmax_plt=[17,17]
n2xmin_select=[15,12]; n2xmax_select=[17,16]
xunit=yu.ens2a[ens]; yunit=yu.ens2aInv[ens]/1000
for i in range(2):
    xmin_plt=n2xmin_plt[i]; xmax_plt=n2xmax_plt[i]
    xmin_select=n2xmin_select[i]; xmax_select=n2xmax_select[i]
    t=evals[:,:,i]
    t=-np.log(t)/dt
    mean,err=yu.jackme(t)
    plt_x=ts[xmin_plt:xmax_plt]*xunit; plt_y=mean[xmin_plt:xmax_plt]*yunit; plt_yerr=err[xmin_plt:xmax_plt]*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color=yu.colors8[i])
    plt_x=ts[xmin_select:xmax_select]*xunit; plt_y=mean[xmin_select:xmax_select]*yunit; plt_yerr=err[xmin_select:xmax_select]*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color=yu.colors8[i],mfc='white')
ax.set_xlim([0,1.7])

ax=axs[0,1]

n2EN_selected={}
for i in range(2):
    xmin_select=n2xmin_select[i]; xmax_select=n2xmax_select[i]
    color=yu.colors8[i]
    t=evals[:,:,i]
    t=-np.log(t)/dt
    xmins=np.arange(1,xmax_select-1)
    fits=yu.doFits_const(t,xmins,[xmax_select],corrQ=False)
    for fit in fits:
        (xmin,xmax),pars_jk,chi2_jk,Ndof = fit
        mean,err=yu.jackme(pars_jk)
        plt_x=ts[xmin]*xunit; plt_y=mean[0]*yunit; plt_yerr=err[0]*yunit
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white' if xmin==xmin_select else None)
        
        if xmin==xmin_select:
            n2EN_selected[i]=pars_jk[:,0]
            ax.axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y,plt_yerr))
            axs[0,0].axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2)
print(yu.jackme_un2str(n2EN_selected[0]*yunit*1000),yu.jackme_un2str(n2EN_selected[1]*yunit*1000),yu.jackme_un2str((n2EN_selected[1]-n2EN_selected[0])*yunit*1000))
ax.legend()

xmin_plt=1; xmax_plt=16
ax=axs[1,0]; yunit=1; color='r'
xmin_select=7; xmax_select=12
ax.set_ylim([-0.02,0.05])
t=np.real(evecs[:,:,0,1])/np.real(evecs[:,:,0,0])
mean,err=yu.jackme(t)
xmin=1
plt_x=ts[xmin_plt:xmax_plt]*xunit; plt_y=mean[xmin_plt:xmax_plt]*yunit; plt_yerr=err[xmin_plt:xmax_plt]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
plt_x=ts[xmin_select:xmax_select]*xunit; plt_y=mean[xmin_select:xmax_select]*yunit; plt_yerr=err[xmin_select:xmax_select]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white')

ax=axs[1,1]
xmins=np.arange(1,xmax_select-1)
fits=yu.doFits_const(t,xmins,[xmax_select],corrQ=False)
for fit in fits:
    (xmin,xmax),pars_jk,chi2_jk,Ndof = fit
    mean,err=yu.jackme(pars_jk)
    plt_x=ts[xmin]*xunit; plt_y=mean[0]*yunit; plt_yerr=err[0]*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white' if xmin==xmin_select else None)
    
    if xmin==xmin_select:
        v_selected=pars_jk[:,0]
        ax.axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y,plt_yerr))
        axs[1,0].axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2)
ax.legend()

ax=axs[2,0]; yunit=1
# ax.set_ylim([-0.01,0.01])
t=1/(np.real(evecs[:,:,0,0])*np.real(evecsInv[:,:,0,0]))-1
mean,err=yu.jackme(t)
xmin=1
plt_x=ts[xmin_plt:xmax_plt]*xunit; plt_y=mean[xmin_plt:xmax_plt]*yunit; plt_yerr=err[xmin_plt:xmax_plt]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
plt_x=ts[xmin_select:xmax_select]*xunit; plt_y=mean[xmin_select:xmax_select]*yunit; plt_yerr=err[xmin_select:xmax_select]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white')

ax=axs[2,1]
xmins=np.arange(1,xmax_select-1)
fits=yu.doFits_const(t,xmins,[xmax_select],corrQ=False)
for fit in fits:
    (xmin,xmax),pars_jk,chi2_jk,Ndof = fit
    mean,err=yu.jackme(pars_jk)
    plt_x=ts[xmin]*xunit; plt_y=mean[0]*yunit; plt_yerr=err[0]*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white' if xmin==xmin_select else None)
    
    if xmin==xmin_select:
        w_selected=pars_jk[:,0]
        ax.axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y,plt_yerr))
        axs[2,0].axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2)
ax.legend()

axs[0,0].set_ylabel(r'$E_{eff}$')
axs[1,0].set_ylabel(r'$v_{01}/v_{00}$')
axs[2,0].set_ylabel(r'W')

yu.finalizePlot('GEVP')
yu.save_pkl_reg('vw',[v_selected,w_selected])

956.9(2.1) 1281(38) 324(38)
